In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    FloatSlider,
    IntSlider,
    interactive_output,
    VBox,
    HBox,
    Layout,
    HTMLMath
)

from IPython.display import display, Markdown

# ============================================================
# RANDOM VECTORS AND COVARIANCE MATRIX
# ============================================================

display(Markdown(r"""
## Random Vectors and the Covariance Matrix

A random vector combines several random variables into a single
vector-valued quantity. Its covariance matrix describes both the
variance of each component and the statistical dependence between
different components.

For a two-dimensional random vector
$\mathbf{X}=[X_1,X_2]^T$, the shape and orientation of the point cloud
are directly related to the covariance matrix.
"""))

# ============================================================
# CONTROLS
# ============================================================

sigma1_slider = FloatSlider(
    value=1.0,
    min=0.5,
    max=3.0,
    step=0.1,
    description='σ₁:',
    continuous_update=True,
    layout=Layout(width='300px')
)

sigma2_slider = FloatSlider(
    value=2.0,
    min=0.5,
    max=3.0,
    step=0.1,
    description='σ₂:',
    continuous_update=True,
    layout=Layout(width='300px')
)

rho_slider = FloatSlider(
    value=0.0,
    min=-0.95,
    max=0.95,
    step=0.05,
    description='ρ:',
    continuous_update=True,
    layout=Layout(width='300px')
)

N_slider = IntSlider(
    value=800,
    min=100,
    max=3000,
    step=100,
    description='Samples:',
    continuous_update=True,
    layout=Layout(width='300px')
)

# ------------------------------------------------------------
# Two-column control layout
# ------------------------------------------------------------

controls = HBox(
    [
        VBox(
            [
                sigma1_slider,
                sigma2_slider
            ],
            layout=Layout(
                width='320px'
            )
        ),

        VBox(
            [
                rho_slider,
                N_slider
            ],
            layout=Layout(
                width='320px'
            )
        )
    ],
    layout=Layout(
        width='680px',
        gap='25px',
        align_items='flex-start'
    )
)

display(controls)

# ============================================================
# INTERACTIVE DEMONSTRATION
# ============================================================

def random_vector_demo(sigma1, sigma2, rho, N):

    # --------------------------------------------------------
    # Mean vector
    # --------------------------------------------------------

    mu = np.array([
        0.0,
        0.0
    ])

    # --------------------------------------------------------
    # Covariance matrix
    # --------------------------------------------------------

    covariance = np.array([
        [
            sigma1**2,
            rho * sigma1 * sigma2
        ],
        [
            rho * sigma1 * sigma2,
            sigma2**2
        ]
    ])

    # --------------------------------------------------------
    # Reproducible random experiment
    # --------------------------------------------------------

    rng = np.random.default_rng(
        12345
    )

    samples = rng.multivariate_normal(
        mean=mu,
        cov=covariance,
        size=N
    )

    X1 = samples[:, 0]
    X2 = samples[:, 1]

    # --------------------------------------------------------
    # Sample statistics
    # --------------------------------------------------------

    sample_mean = np.mean(
        samples,
        axis=0
    )

    sample_covariance = np.cov(
        samples,
        rowvar=False,
        ddof=1
    )

    sample_rho = np.corrcoef(
        X1,
        X2
    )[0, 1]

    # ========================================================
    # FIGURE
    # ========================================================

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5)
    )

    # --------------------------------------------------------
    # LEFT — Scatter plot
    # --------------------------------------------------------

    axes[0].scatter(
        X1,
        X2,
        s=14,
        alpha=0.45
    )

    axes[0].axhline(
        0,
        linewidth=1,
        linestyle=':'
    )

    axes[0].axvline(
        0,
        linewidth=1,
        linestyle=':'
    )

    axes[0].set_xlim(
        -9,
        9
    )

    axes[0].set_ylim(
        -9,
        9
    )

    axes[0].set_aspect(
        'equal',
        adjustable='box'
    )

    axes[0].set_xlabel(
        '$X_1$'
    )

    axes[0].set_ylabel(
        '$X_2$'
    )

    axes[0].set_title(
        'Samples of the Random Vector'
    )

    axes[0].grid(
        True,
        alpha=0.25
    )

    # --------------------------------------------------------
    # RIGHT — Covariance ellipse
    # --------------------------------------------------------

    eigenvalues, eigenvectors = np.linalg.eigh(
        covariance
    )

    order = np.argsort(
        eigenvalues
    )[::-1]

    eigenvalues = eigenvalues[
        order
    ]

    eigenvectors = eigenvectors[
        :,
        order
    ]

    theta = np.linspace(
        0,
        2*np.pi,
        600
    )

    unit_circle = np.vstack(
        (
            np.cos(theta),
            np.sin(theta)
        )
    )

    # 2-sigma covariance ellipse

    ellipse = (
        2
        *
        eigenvectors
        @ np.diag(
            np.sqrt(
                eigenvalues
            )
        )
        @ unit_circle
    )

    axes[1].plot(
        ellipse[0, :],
        ellipse[1, :],
        linewidth=2.5
    )

    # --------------------------------------------------------
    # Principal directions
    # --------------------------------------------------------

    for i in range(2):

        direction = (
            2
            *
            np.sqrt(
                eigenvalues[i]
            )
            *
            eigenvectors[:, i]
        )

        axes[1].plot(
            [
                -direction[0],
                direction[0]
            ],
            [
                -direction[1],
                direction[1]
            ],
            linewidth=2
        )

    axes[1].axhline(
        0,
        linewidth=1,
        linestyle=':'
    )

    axes[1].axvline(
        0,
        linewidth=1,
        linestyle=':'
    )

    axes[1].set_xlim(
        -9,
        9
    )

    axes[1].set_ylim(
        -9,
        9
    )

    axes[1].set_aspect(
        'equal',
        adjustable='box'
    )

    axes[1].set_xlabel(
        '$X_1$'
    )

    axes[1].set_ylabel(
        '$X_2$'
    )

    axes[1].set_title(
        'Covariance Ellipse and Principal Directions'
    )

    axes[1].grid(
        True,
        alpha=0.25
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # ========================================================
    # RESULTS — ALL IN ONE HORIZONTAL ROW
    # ========================================================

    theoretical_matrix = HTMLMath(
        value=(
            r'\('
            r'\mathbf{C}_{X}'
            r'='
            r'\begin{bmatrix}'
            + f'{covariance[0,0]:.4f}'
            + r'&'
            + f'{covariance[0,1]:.4f}'
            + r'\\'
            + f'{covariance[1,0]:.4f}'
            + r'&'
            + f'{covariance[1,1]:.4f}'
            + r'\end{bmatrix}'
            r'\)'
        ),
        layout=Layout(
            width='230px'
        )
    )

    rho_theoretical = HTMLMath(
        value=(
            r'\('
            r'\rho_{12}'
            r'='
            + f'{rho:.4f}'
            + r'\)'
        ),
        layout=Layout(
            width='140px'
        )
    )

    mean_estimated = HTMLMath(
        value=(
            r'\('
            r'\widehat{\boldsymbol{\mu}}'
            r'='
            r'\begin{bmatrix}'
            + f'{sample_mean[0]:.4f}'
            + r'\\'
            + f'{sample_mean[1]:.4f}'
            + r'\end{bmatrix}'
            r'\)'
        ),
        layout=Layout(
            width='180px'
        )
    )

    covariance_estimated = HTMLMath(
        value=(
            r'\('
            r'\widehat{\mathbf{C}}_{X}'
            r'='
            r'\begin{bmatrix}'
            + f'{sample_covariance[0,0]:.4f}'
            + r'&'
            + f'{sample_covariance[0,1]:.4f}'
            + r'\\'
            + f'{sample_covariance[1,0]:.4f}'
            + r'&'
            + f'{sample_covariance[1,1]:.4f}'
            + r'\end{bmatrix}'
            r'\)'
        ),
        layout=Layout(
            width='245px'
        )
    )

    rho_estimated = HTMLMath(
        value=(
            r'\('
            r'\widehat{\rho}_{12}'
            r'='
            + f'{sample_rho:.4f}'
            + r'\)'
        ),
        layout=Layout(
            width='160px'
        )
    )

    results_row = HBox(
        [
            theoretical_matrix,
            rho_theoretical,
            mean_estimated,
            covariance_estimated,
            rho_estimated
        ],
        layout=Layout(
            width='1100px',
            gap='18px',
            align_items='center',
            justify_content='flex-start'
        )
    )

    display(
        results_row
    )


# ============================================================
# CONNECT CONTROLS
# ============================================================

output = interactive_output(
    random_vector_demo,
    {
        'sigma1': sigma1_slider,
        'sigma2': sigma2_slider,
        'rho': rho_slider,
        'N': N_slider
    }
)

display(
    output
)